In [4]:
import numpy as np

def monte_carlo_european_price(S0, K, T, r, sigma, num_simulations, option_type='call'):
    """
    Prices European option using a vectorised Monte Carlo simulation.
    
    Parameters:
    S0 (float): Initial stock price
    K (float): Strike Price
    T (float): Time to maturity in years
    r (float): Risk-free interest rate
    sigma (float): Volatility of the underlying asset
    num_simulations (int): Number of simulations
    option_type (str): 'call' or 'put'
    
    Returns:
    tuple: (Estimated Option Price, 95% Confidence Interval Half-Width)
    """

    # 1. Generate standard normal random variables (Z ~ N(0,1))
    # Use a fixed random seed for reproducability 
    rng = np.random.default_rng() 
    Z = rng.standard_normal(num_simulations) 

    # 2. Simulate terminal asset prices at time T using GBM
    # S(T) = S(0) * exp((r - 0.5*sigma^2) * T + sigma * sqret(T) * Z)
    drift = (r - 0.5*sigma**2) * T
    diffusion = sigma * np.sqrt(T) * Z 
    ST = S0 * np.exp(drift + diffusion) 

    # 3. Calculate expected payoff
    if option_type.lower() == 'call':
        payoffs = np.maximum(ST-K, 0.0)
    elif option_type.lower() == 'put':
        payoffs = np.maximum(0.0, K-ST)
    else:
        raise ValueError("option_type must be 'call' or 'put'")

    # 4. Discount the payoffs at maturity
    discount_factor = np.exp(-r*T)
    discounted_payoffs = payoffs * discount_factor 

    # 5. Statistic Outputs
    price = np.mean(discounted_payoffs) 
    standard_error = np.std(discounted_payoffs) / np.sqrt(num_simulations) 

    # 95% Confidence Interval is roughly +/- 1.96 * Standard Error 
    confidence_interval_width = 1.96*standard_error

    return float(price), float(confidence_interval_width)

if __name__ == "__main__":

    # Inputs 
    S0 = 100.00
    K = 105.0
    T = 1.0 
    r = 0.05 
    sigma = 0.20 
    N = 1000 

    option_price, ci = monte_carlo_european_price(S0, K, T, r, sigma, N, 'call') 

    print(f"Estimated Call Option Price: {option_price:.4f}") 
    print(f"95% Confidence Interval [{option_price - ci:.4f}, {option_price + ci:.4g}]")

Estimated Call Option Price: 8.3129
95% Confidence Interval [7.4648, 9.161]


## Notes:

- Memory Architecture Bottlenecks
    - Problem: A large num_sim creates large congiuous block of mem, this risks hitting a memory wall or triggering cache misses
    - Fix: Batch the simulations, keeping an accumating running sum and variance
- Bypassing the Python GIL (Global Intepreter Lock):
    - Problem: Standard python multi-threading will not speed up simulation. GIL is execute on a single core, making task CPU-bound
    - Fix: Use multiprocessing to spin up separate OS processes. 


## Practise

In [ ]:
import numpy as np

def monte_carlo_euro_option_price(S0, K, T, r, sigma, num_sims = 10000, option_type = 'call'):

    # Description section

    # 1. Random seed & reproducability 
    rng = np.random.default_rng()
    Z = rng.standard_normal(num_sims)

    # 2. Drift + Diffusion
    drift = (r * 0.5*sigma**2) * T
    diffusion = sigma * np.sqrt(T) * Z 
    ST = S0 * np.exp(drift + diffusion) 

    # 3. Payoff
    if option_type == 'call':
        payoffs = max(ST - K, 0)
    elif option_type == 'put':
        payoffs = max(0, K - ST)
    else:
        raise ValueError("Option_type must be a string of 'call' or 'put'.")

    # 4. Discount the payoffs
    discount_factor = np.exp(-r*T)
    discounted_payoffs = payoffs * discount_factor

    # 5. Option Price &  Standard Error 
    option_price = np.mean(discounted_payoffs)
    standard_error = np.std(discounted_payoffs) / np.sqrt(num_sims) 

    # 6. Confidence intervals
    ci_width = 1.96 * standard_error

    return float(option_price), float(ci_width)


if __name__ == "__main__":

    S0 = 100.00             # Current stock price
    K = 105.00              # Strike price
    T = 1                   # Time of maturity/expiry
    r = 0.05                # Risk-free rate
    sigma = 0.20            # Volatility
    num_sims = 10000         # Number of simulations
    option_type = 'call'

    option_price, ci = monte_carlo_euro_option_price(S0, K, T, r, sigma, option_type)

    print("Estimated cost of option: £{option_price:.4f}")
    print("Modelling confidence interval, 95%: [{option_price - ci:.4f},{option_price + ci:.4f}]")